In [2]:
%%writefile app1.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import requests

from prophet import Prophet

# -------------------------------------------------
# Genel Ayarlar
# -------------------------------------------------
st.set_page_config(page_title="CO₂ Emisyon Dashboard – OWID + Prophet + Wikipedia", layout="wide")

DATA_PATH = "owid-co2-data.csv"
TARGET_LAST_YEAR = 2024   # OWID için modele kadar kullanılacak son gerçek yıl
BASELINE_YEAR = 2000      # OWID artış/azalış ve scatter için başlangıç yılı
FORECAST_HORIZON = 2050   # OWID -> Prophet tahmin son yılı

WIKI_URL = "https://en.wikipedia.org/wiki/List_of_countries_by_carbon_dioxide_emissions_per_capita"
WIKI_HEADERS = {"User-Agent": "Mozilla/5.0"}

# -------------------------------------------------
# OWID Veri Yükleme
# -------------------------------------------------
@st.cache_data
def load_owid_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    # Sadece gerçek ülke satırları (World, Asia vs. hariç)
    df = df[df["iso_code"].notna()].copy()

    cols = [
        "country", "iso_code", "year",
        "co2", "co2_per_capita",
        "population", "gdp",
        "total_ghg", "total_ghg_excluding_lucf"
    ]
    existing = [c for c in cols if c in df.columns]
    df = df[existing]

    df = df.sort_values(["country", "year"])
    return df

df = load_owid_data(DATA_PATH)
countries = sorted(df["country"].unique())
min_year = int(df["year"].min())
max_year = int(df["year"].max())
latest_year = min(TARGET_LAST_YEAR, max_year)

# -------------------------------------------------
# Wikipedia (Scraping) Veri Yükleme
# -------------------------------------------------
def _flatten_multiindex_columns(cols):
    # MultiIndex ise "A_B" gibi düzleştir
    if isinstance(cols, pd.MultiIndex):
        return ["_".join([str(x) for x in tup if str(x) != "nan"]).strip() for tup in cols]
    return [str(c) for c in cols]

def _extract_year_from_col(colname: str):
    m = re.search(r"(18\d{2}|19\d{2}|20\d{2})", colname)
    return int(m.group(1)) if m else None

@st.cache_data
def load_wikipedia_data(url: str) -> pd.DataFrame:
    r = requests.get(url, headers=WIKI_HEADERS, timeout=30)
    r.raise_for_status()

    # Wikipedia sayfasındaki ilk büyük tablo genelde aradığımız tablo oluyor
    tables = pd.read_html(r.text, header=[0, 1])
    if not tables:
        raise ValueError("Wikipedia tablosu bulunamadı.")

    raw = tables[0].copy()
    raw.columns = _flatten_multiindex_columns(raw.columns)

    # Ülke sütununu yakala (Location veya Country benzeri)
    # Wikipedia bu sayfada genelde "Location_Location" olur.
    country_col = None
    for c in raw.columns:
        if "Location" in c or "Country" in c:
            country_col = c
            break
    if country_col is None:
        # fallback: ilk sütun
        country_col = raw.columns[0]

    # Yıl sütunlarını topla
    year_cols = []
    year_map = {}
    for c in raw.columns:
        y = _extract_year_from_col(c)
        if y is not None and ("Emissions" in c or "tons" in c or "per capita" in c or "year" in c):
            year_cols.append(c)
            year_map[c] = y

    if not year_cols:
        # bazı durumlarda year_cols regex ile yakalanamayabilir; tüm kolonlardan yıl çekmeyi dene
        for c in raw.columns:
            y = _extract_year_from_col(c)
            if y is not None:
                year_cols.append(c)
                year_map[c] = y

    if not year_cols:
        raise ValueError("Wikipedia tablosunda yıl sütunları bulunamadı.")

    # En güncel yıl ve baseline yıl (2000 varsa) seç
    years_found = sorted(set(year_map.values()))
    latest_y = max(years_found)

    # 2000 varsa onu kullan, yoksa en yakın alt yıl
    if 2000 in years_found:
        base_y = 2000
    else:
        # en yakın 2000 altı değilse en küçük değil; 2000'e en yakın yıl
        base_y = min(years_found, key=lambda x: abs(x - 2000))

    # İlgili kolon adlarını bul
    latest_col = [c for c in year_cols if year_map.get(c) == latest_y][0]
    base_col = [c for c in year_cols if year_map.get(c) == base_y][0]

    out = raw[[country_col, base_col, latest_col]].copy()
    out.columns = ["Country", f"Emissions_{base_y}", f"Emissions_{latest_y}"]

    # Sayısallaştır
    for c in [f"Emissions_{base_y}", f"Emissions_{latest_y}"]:
        out[c] = (
            out[c].astype(str)
            .str.replace("−", "-")
            .str.replace("—", "")
            .str.replace("–", "-")
            .str.replace(",", "")
            .str.replace("%", "")
            .str.strip()
        )
        out[c] = pd.to_numeric(out[c], errors="coerce")

    out = out.dropna(subset=[f"Emissions_{latest_y}"]).copy()
    out["Abs_Change"] = out[f"Emissions_{latest_y}"] - out[f"Emissions_{base_y}"]
    out["Pct_Change"] = np.where(
        out[f"Emissions_{base_y}"] != 0,
        out["Abs_Change"] / out[f"Emissions_{base_y}"] * 100,
        np.nan
    )

    return out, base_y, latest_y

# -------------------------------------------------
# OWID Yardımcı Fonksiyonlar
# -------------------------------------------------
def get_country_series(country: str) -> pd.DataFrame:
    cdf = df[df["country"] == country].copy()
    cdf = cdf.sort_values("year")
    cdf = cdf[cdf["co2_per_capita"].notna()]
    if cdf.empty:
        return cdf
    upper = min(TARGET_LAST_YEAR, int(cdf["year"].max()))
    cdf = cdf[cdf["year"] <= upper]
    return cdf

def fit_prophet_for_country(cdf: pd.DataFrame) -> Prophet:
    dfp = cdf[["year", "co2_per_capita"]].copy()
    dfp["ds"] = pd.to_datetime(dfp["year"].astype(str) + "-01-01")
    dfp["y"] = dfp["co2_per_capita"].astype(float)

    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False
    )
    m.fit(dfp[["ds", "y"]])
    return m

def forecast_to_year(model: Prophet, last_real_year: int, horizon_year: int) -> pd.DataFrame:
    periods = max(0, horizon_year - last_real_year)
    future = model.make_future_dataframe(periods=periods, freq="Y")
    forecast = model.predict(future)
    forecast["year"] = forecast["ds"].dt.year
    return forecast

# -------------------------------------------------
# Sidebar Menü
# -------------------------------------------------
menu = st.sidebar.selectbox(
    "📌 Sayfa Seç",
    [
        "Ana Sayfa (OWID)",
        "Ülke Karşılaştırma (OWID - Gerçek Veriler)",
        "Top 10 Artış (OWID 2000–2024)",
        "Top 10 Azalış (OWID 2000–2024)",
        "Dağılım Grafiği (OWID 2000 vs 2024)",
        "Tahmin Grafiği (OWID Tek Ülke – Prophet)",
        "Wikipedia Verileri (Scraping)"
    ]
)

st.sidebar.write("---")
st.sidebar.write(f"OWID veri yılları: {min_year} – {max_year}")
st.sidebar.write(f"OWID model eğitimi son yıl: {latest_year}")
st.sidebar.write(f"OWID tahmin ufku: {FORECAST_HORIZON}")
st.sidebar.write("Tahmin modeli: Prophet (sadece OWID tek ülke sayfasında)")

# -------------------------------------------------
# 1) Ana Sayfa (OWID)
# -------------------------------------------------
if menu == "Ana Sayfa (OWID)":
    st.title("🌍 Küresel CO₂ Emisyon Dashboard – OWID")

    latest = df[(df["year"] == latest_year) & (df["co2_per_capita"].notna())]

    col1, col2, col3 = st.columns(3)
    col1.metric("Ülke Sayısı", latest["country"].nunique())
    col2.metric("Ort. Kişi Başı CO₂", round(latest["co2_per_capita"].mean(), 3))
    col3.metric(
        "Toplam CO₂",
        round(latest["co2"].sum(), 2) if "co2" in latest.columns else "Veri yok"
    )

    st.subheader(f"⬆️ En Yüksek 10 Ülke ({latest_year}, kişi başı CO₂)")
    st.dataframe(
        latest.sort_values("co2_per_capita", ascending=False)
        .head(10)[["country", "co2_per_capita"]]
    )

    st.subheader(f"⬇️ En Düşük 10 Ülke ({latest_year}, kişi başı CO₂)")
    st.dataframe(
        latest.sort_values("co2_per_capita", ascending=True)
        .head(10)[["country", "co2_per_capita"]]
    )

# -------------------------------------------------
# 2) Ülke Karşılaştırma (OWID - tahminsiz)
# -------------------------------------------------
elif menu == "Ülke Karşılaştırma (OWID - Gerçek Veriler)":
    st.title("🏳️ Ülke Karşılaştırma – OWID (Tahmin Yok)")

    selected_countries = st.multiselect(
        "Karşılaştırmak istediğin ülkeleri seç:",
        countries,
        default=["Turkey"] if "Turkey" in countries else countries[:3]
    )

    start_year, end_year = st.slider(
        "Gösterilecek yıl aralığı:",
        min_year,
        latest_year,
        (max(min_year, 2000), latest_year)
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    for c in selected_countries:
        cdf = get_country_series(c)
        cdf = cdf[(cdf["year"] >= start_year) & (cdf["year"] <= end_year)]
        if cdf.empty:
            continue
        ax.plot(cdf["year"], cdf["co2_per_capita"], marker="o", label=c)

    ax.set_xlabel("Yıl")
    ax.set_ylabel("Kişi Başı CO₂ (ton)")
    ax.set_title("Seçilen Ülkelerde Kişi Başı CO₂ – OWID GERÇEK Değerler")
    ax.legend()
    st.pyplot(fig)

# -------------------------------------------------
# 3) Top 10 Artış (OWID 2000–2024)
# -------------------------------------------------
elif menu == "Top 10 Artış (OWID 2000–2024)":
    st.title("📈 OWID: 2000–2024 Arası Kişi Başı CO₂ En Fazla Artış Gösteren 10 Ülke")

    subset = df[df["year"].isin([BASELINE_YEAR, latest_year])]
    pivot = subset.pivot(index="country", columns="year", values="co2_per_capita")
    pivot = pivot.dropna(subset=[BASELINE_YEAR, latest_year])

    pivot["abs_change"] = pivot[latest_year] - pivot[BASELINE_YEAR]
    pivot["pct_change"] = pivot["abs_change"] / pivot[BASELINE_YEAR] * 100

    top10 = pivot.sort_values("pct_change", ascending=False).head(10)

    st.dataframe(
        top10[["pct_change", "abs_change", BASELINE_YEAR, latest_year]]
        .rename(columns={BASELINE_YEAR: f"value_{BASELINE_YEAR}", latest_year: f"value_{latest_year}"})
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(top10.index, top10["pct_change"])
    ax.set_xlabel("Yüzde Değişim (%)")
    ax.set_title(f"OWID {BASELINE_YEAR}–{latest_year} Kişi Başı CO₂ Yüzde Artışı – Top 10")
    ax.invert_yaxis()
    st.pyplot(fig)

# -------------------------------------------------
# 4) Top 10 Azalış (OWID 2000–2024)
# -------------------------------------------------
elif menu == "Top 10 Azalış (OWID 2000–2024)":
    st.title("📉 OWID: 2000–2024 Arası Kişi Başı CO₂ En Fazla Azalan 10 Ülke")

    subset = df[df["year"].isin([BASELINE_YEAR, latest_year])]
    pivot = subset.pivot(index="country", columns="year", values="co2_per_capita")
    pivot = pivot.dropna(subset=[BASELINE_YEAR, latest_year])

    pivot["abs_change"] = pivot[latest_year] - pivot[BASELINE_YEAR]
    pivot["pct_change"] = pivot["abs_change"] / pivot[BASELINE_YEAR] * 100

    bottom10 = pivot.sort_values("pct_change", ascending=True).head(10)

    st.dataframe(
        bottom10[["pct_change", "abs_change", BASELINE_YEAR, latest_year]]
        .rename(columns={BASELINE_YEAR: f"value_{BASELINE_YEAR}", latest_year: f"value_{latest_year}"})
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(bottom10.index, bottom10["pct_change"])
    ax.set_xlabel("Yüzde Değişim (%)")
    ax.set_title(f"OWID {BASELINE_YEAR}–{latest_year} Kişi Başı CO₂ Yüzde Azalışı – Top 10")
    ax.invert_yaxis()
    st.pyplot(fig)

# -------------------------------------------------
# 5) Dağılım Grafiği (OWID 2000 vs 2024)
# -------------------------------------------------
elif menu == "Dağılım Grafiği (OWID 2000 vs 2024)":
    st.title("🟣 OWID: 2000 vs 2024 Kişi Başı CO₂ Dağılımı")

    subset = df[df["year"].isin([BASELINE_YEAR, latest_year])]
    pivot = subset.pivot(index="country", columns="year", values="co2_per_capita")
    pivot = pivot.dropna(subset=[BASELINE_YEAR, latest_year])

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(pivot[BASELINE_YEAR], pivot[latest_year])
    ax.set_xlabel(f"{BASELINE_YEAR} Kişi Başı CO₂ (ton)")
    ax.set_ylabel(f"{latest_year} Kişi Başı CO₂ (ton)")
    ax.set_title(f"OWID {BASELINE_YEAR} vs {latest_year} – Ülke Bazlı")

    min_val = min(pivot[BASELINE_YEAR].min(), pivot[latest_year].min())
    max_val = max(pivot[BASELINE_YEAR].max(), pivot[latest_year].max())
    ax.plot([min_val, max_val], [min_val, max_val], linestyle="--")
    st.pyplot(fig)

    st.write("Veri tablosu:")
    st.dataframe(
        pivot.reset_index()
        .rename(columns={BASELINE_YEAR: f"value_{BASELINE_YEAR}", latest_year: f"value_{latest_year}"})
    )

# -------------------------------------------------
# 6) Tahmin Grafiği (OWID Tek Ülke – Prophet)
# -------------------------------------------------
elif menu == "Tahmin Grafiği (OWID Tek Ülke – Prophet)":
    st.title("🔮 OWID: Tek Ülke CO₂ Tahmini – Prophet")

    country = st.selectbox(
        "Ülke seç:",
        countries,
        index=countries.index("Turkey") if "Turkey" in countries else 0
    )

    cdf = get_country_series(country)

    if cdf.shape[0] < 5:
        st.warning("Bu ülke için Prophet kullanmak için yeterli veri yok (en az 5 yıl önerilir).")
    else:
        model = fit_prophet_for_country(cdf)
        last_real_year = int(cdf["year"].max())
        forecast = forecast_to_year(model, last_real_year, horizon_year=FORECAST_HORIZON)

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.scatter(cdf["year"], cdf["co2_per_capita"], color="red", label="Gerçek Veri (OWID)")
        ax.plot(forecast["year"], forecast["yhat"], label="Prophet Tahmini (yhat)")
        ax.fill_between(forecast["year"], forecast["yhat_lower"], forecast["yhat_upper"], alpha=0.2, label="Tahmin Aralığı")
        ax.axvline(x=last_real_year, linestyle="--", label=f"Son Gerçek Yıl: {last_real_year}")
        ax.set_xlabel("Yıl")
        ax.set_ylabel("Kişi Başı CO₂ (ton)")
        ax.set_title(f"{country} – Prophet ile {FORECAST_HORIZON}'e Kadar Tahmin (OWID)")
        ax.legend()
        st.pyplot(fig)

        # 2050 NaN olmasın diye: modeli direkt çağırıp 2030/2040/2050 tahmini alıyoruz
        years_to_check = [2030, 2040, FORECAST_HORIZON]
        future_df = pd.DataFrame({"ds": pd.to_datetime([f"{y}-01-01" for y in years_to_check])})
        future_pred = model.predict(future_df)
        future_pred["year"] = future_pred["ds"].dt.year

        last_real_value = float(cdf[cdf["year"] == last_real_year]["co2_per_capita"].iloc[0])
        pred_2050 = float(future_pred.loc[future_pred["year"] == FORECAST_HORIZON, "yhat"].iloc[0])

        col1, col2, col3 = st.columns(3)
        col1.metric("Son Gerçek Yıl", last_real_year)
        col2.metric("Son Gerçek Değer", round(last_real_value, 3))
        col3.metric(f"{FORECAST_HORIZON} Tahmini (yhat)", round(pred_2050, 3))

        st.subheader("Örnek Tahmin Noktaları")
        for _, r in future_pred.iterrows():
            st.write(
                f"- {int(r['year'])}: yhat={r['yhat']:.3f} "
                f"(alt={r['yhat_lower']:.3f}, üst={r['yhat_upper']:.3f})"
            )

# -------------------------------------------------
# 7) Wikipedia Verileri (Scraping)
# -------------------------------------------------
elif menu == "Wikipedia Verileri (Scraping)":
    st.title("📚 Wikipedia Verileri (Web Scraping) – Kişi Başı CO₂")

    st.write(
        "Bu sayfa Wikipedia'dan **web scraping** ile çekilen verileri gösterir. "
        "OWID ile karıştırmıyoruz: Bu sayfa sadece Wikipedia tablosunun görselleştirmesi."
    )

    try:
        wiki_df, wiki_base_year, wiki_latest_year = load_wikipedia_data(WIKI_URL)
    except Exception as e:
        st.error(f"Wikipedia verisi alınamadı: {e}")
        st.stop()

    colA, colB, colC = st.columns(3)
    colA.metric("Wikipedia Ülke Sayısı", wiki_df["Country"].nunique())
    colB.metric("Baseline Yıl", wiki_base_year)
    colC.metric("En Güncel Yıl", wiki_latest_year)

    # Top 10 / Bottom 10 (en güncel yıl)
    st.subheader(f"⬆️ Wikipedia Top 10 (Kişi Başı CO₂, {wiki_latest_year})")
    top10 = wiki_df.sort_values(f"Emissions_{wiki_latest_year}", ascending=False).head(10)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(top10["Country"], top10[f"Emissions_{wiki_latest_year}"])
    ax.set_xlabel("ton/kişi")
    ax.set_title(f"Wikipedia – En Yüksek 10 Ülke ({wiki_latest_year})")
    ax.invert_yaxis()
    st.pyplot(fig)
    st.dataframe(top10[["Country", f"Emissions_{wiki_base_year}", f"Emissions_{wiki_latest_year}", "Pct_Change", "Abs_Change"]])

    st.subheader(f"⬇️ Wikipedia Bottom 10 (Kişi Başı CO₂, {wiki_latest_year})")
    bottom10 = wiki_df.sort_values(f"Emissions_{wiki_latest_year}", ascending=True).head(10)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(bottom10["Country"], bottom10[f"Emissions_{wiki_latest_year}"])
    ax.set_xlabel("ton/kişi")
    ax.set_title(f"Wikipedia – En Düşük 10 Ülke ({wiki_latest_year})")
    ax.invert_yaxis()
    st.pyplot(fig)
    st.dataframe(bottom10[["Country", f"Emissions_{wiki_base_year}", f"Emissions_{wiki_latest_year}", "Pct_Change", "Abs_Change"]])

    # Seçili ülkeler için 2000 -> güncel yıl mini trend (2 nokta)
    st.subheader(f"📌 Seçili Ülkeler: {wiki_base_year} → {wiki_latest_year} Değişimi (Wikipedia)")

    selected = st.multiselect(
        "Ülke seç (Wikipedia listesi):",
        sorted(wiki_df["Country"].unique()),
        default=["Turkey"] if "Turkey" in set(wiki_df["Country"]) else list(wiki_df["Country"].head(3))
    )

    sel_df = wiki_df[wiki_df["Country"].isin(selected)].copy()
    if sel_df.empty:
        st.warning("Seçilen ülkeler Wikipedia tablosunda bulunamadı.")
    else:
        fig, ax = plt.subplots(figsize=(10, 5))
        for _, r in sel_df.iterrows():
            ax.plot([wiki_base_year, wiki_latest_year],
                    [r[f"Emissions_{wiki_base_year}"], r[f"Emissions_{wiki_latest_year}"]],
                    marker="o", label=r["Country"])
        ax.set_xlabel("Yıl")
        ax.set_ylabel("ton/kişi")
        ax.set_title(f"Wikipedia – {wiki_base_year} → {wiki_latest_year} Kişi Başı CO₂ Değişimi")
        ax.legend()
        st.pyplot(fig)

        st.dataframe(sel_df.sort_values(f"Emissions_{wiki_latest_year}", ascending=False))

    st.subheader("📄 Wikipedia Ham Veri (Temizlenmiş)")
    st.dataframe(wiki_df)


Overwriting app1.py
